In [181]:
import os
from pyoxigraph import Store, RdfFormat
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from kneed import KneeLocator

from elasticsearch import Elasticsearch

from utils import ollama_request, EMBEDD_MODEL_1

import spacy
nlp = spacy.load("en_core_web_sm")

INDEX_NAME = os.getenv("INDEX_NAME")
ENT_INDEX_NAME = f"{INDEX_NAME}_entities_index"
FRAGMENT_INDEX_NAME = f"{INDEX_NAME}_fragments"
TRIPLETS_INDEX_NAME = f"{INDEX_NAME}_triplets_index"

EMBEDDING_MODEL = EMBEDD_MODEL_1

es_client = Elasticsearch('http://localhost:9200')

graph = Store()
graph.load(
    path=f"{INDEX_NAME}.ttl",
    format=RdfFormat.TURTLE
)

In [182]:
def extract_objects(text):
    doc = nlp(text)

    subjects = []
    target_deps = {
        "nsubj",
        "nsubjpass",
        "dobj",
        "pobj",
        "iobj",
    }
    for token in doc:
        if token.dep_ in target_deps:
            subject = " ".join(
                t.text for t in token.subtree
            )
            subjects.append(subject)

    return subjects

In [183]:
def execute_query(graph, start_date, end_date, filtering_criteria):
    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id ?triplet_id
WHERE {{
    ?stmt rdf:reifies <<( ?s ?p ?o )>> ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:triplet_id ?triplet_id ;
          ns1:speech_id ?speech_id .

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
        {filtering_criteria}
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        filtering_criteria=filtering_criteria
    )

    return graph.query(query)

In [184]:
def get_df_from_query_result(query_res):
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "start": [],
        "end": [],
        "speech_id": [],
    }
    for row in query_res:
        fragment_start = row["start"].value
        fragment_end = row["end"].value
        speech_id = row["speech_id"].value

        triplets_resp["subject"].append(row["s"].value.split('/')[-1])
        triplets_resp["predicate"].append(row["p"].value.split('/')[-1])
        triplets_resp["object"].append(row["o"].value.split('/')[-1])
        triplets_resp["date"].append(str(row["date"].value))
        triplets_resp["start"].append(fragment_start)
        triplets_resp["end"].append(fragment_end)
        triplets_resp["speech_id"].append(speech_id)
    return pd.DataFrame(triplets_resp)

In [185]:
def get_triplets_by_id(
    graph,
    triplet_ids,
    start_date="1900-01-01",
    end_date="2100-01-01"
):
    triplet_ids_values = " ".join(str(x) for x in triplet_ids)

    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id
WHERE {{
    ?stmt rdf:reifies <<( ?s ?p ?o )>> ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:speech_id ?speech_id ;
          ns1:triplet_id ?triplet_id .

    VALUES ?triplet_id {{ {triplet_ids_values} }}

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        triplet_ids_values=triplet_ids_values,
    )

    return graph.query(query)

In [186]:
def knee_cutoff(scores):
    scores = np.sort(np.asarray(scores))[::-1]

    indices = np.arange(len(scores))

    knee = KneeLocator(
        indices,
        scores,
        curve="convex",
        direction="decreasing"
    )

    return int(knee.knee) - 1

In [187]:
def score_df_cutoff(df, top_k=10):
    df = df.iloc[:top_k, :].sort_values(by="score", ascending=False)

    sorted_scores = df["score"].values
    if len(sorted_scores) < 2:
        return df
    gaps = sorted_scores[:-1] - sorted_scores[1:]
    cutoff_1 = np.argmax(gaps) + 1
    cutoff_2 = knee_cutoff(sorted_scores)

    if cutoff_2 == 0:
        cutoff = cutoff_1
    else:
        cutoff = cutoff_2
    return df.iloc[:cutoff, :]

In [188]:
def find_similar_entities(entity, score_threshold=0.5, top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(entity)
    final_res = {
        "score": [],
        "value_name": [],
    }
    response = es_client.search(
        index=ENT_INDEX_NAME,
        size=100,
        knn={
            "field": "value_embedding",
            "query_vector": query_embedding.tolist(),
            "k": 100,
            "num_candidates": 100,
            "filter": {
                "match_all": {}
            }
        },
        source=["value_name"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["value_name"].append(result["_source"]["value_name"])
    
    final_res = pd.DataFrame(final_res)
    return score_df_cutoff(final_res, top_k=top_k)


In [189]:
def find_question_related_triplets(graph, question, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01", top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(question)
    final_res = {
        "score": [],
        "triplet_id": [],
    }
    response = es_client.search(
        index=TRIPLETS_INDEX_NAME,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": query_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "match_all": {}
            }
        },
        source=["triplet_id"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    related_triplets = pd.DataFrame(final_res)
    related_triplets = score_df_cutoff(related_triplets, top_k=top_k)
    related_ids = list(related_triplets["triplet_id"])

    query_res = get_triplets_by_id(graph, related_ids, start_date=start_date, end_date=end_date)
    res_df = get_df_from_query_result(query_res)
    res_df["information_type"] = "main"

    return res_df


In [196]:
def find_additional_related_triplets(graph, question, score_threshold=0.5, top_entity_k=10, top_triplet_k=10, start_date="1900-01-01", end_date="2100-01-01"):
    question_objects = extract_objects(question)
    question_embedding = EMBEDDING_MODEL.encode(question)
    similar_entities = []
    for object in question_objects:
        similar_entities = similar_entities + list(find_similar_entities(object, score_threshold=score_threshold, top_k=top_entity_k)["value_name"].values)

    filtering_criteria = "&& (?s = entity:{val} || ?o = entity:{val})"
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "start": [],
        "end": [],
        "speech_id": [],
        "triplet_id": [],
    } 
    for ent in similar_entities:
        try:
            query_res = list(
                execute_query(
                    graph=graph,
                    start_date=start_date,
                    end_date=end_date,
                    filtering_criteria=filtering_criteria.format(val=ent),
                )
            )
            # print(f"Found {len(query_res)} triplets for entity: {ent}")
        except Exception as e:
            continue

        for row in query_res:
            fragment_start = row["start"].value
            fragment_end = row["end"].value
            speech_id = row["speech_id"].value

            triplet_id = row["triplet_id"].value
            
            triplets_resp["subject"].append(row["s"].value.split('/')[-1])
            triplets_resp["predicate"].append(row["p"].value.split('/')[-1])
            triplets_resp["object"].append(row["o"].value.split('/')[-1])
            triplets_resp["date"].append(str(row["date"].value))
            triplets_resp["start"].append(fragment_start)
            triplets_resp["end"].append(fragment_end)
            triplets_resp["speech_id"].append(speech_id)
            triplets_resp["triplet_id"].append(int(triplet_id))

    triplets_df = pd.DataFrame(triplets_resp)
    final_res = {
        "score": [],
        "triplet_id": [],
    }

    response = es_client.search(
        index=TRIPLETS_INDEX_NAME,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": question_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "terms": {
                    "triplet_id": triplets_df["triplet_id"].tolist()
                }
            }
        },
        source=["triplet_id"]
    )
    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    final_scores = pd.DataFrame(final_res)
    final_scores["triplet_id"] = final_scores["triplet_id"].astype("int64")
    triplets_df["triplet_id"] = triplets_df["triplet_id"].astype("int64")

    triplets_df = triplets_df.merge(
        final_scores,
        on="triplet_id",
        how="inner"
    )
    triplets_df = score_df_cutoff(triplets_df, top_k=top_triplet_k)

    triplets_df["information_type"] = "additional"
    triplets_df = triplets_df.drop(columns=["score", "triplet_id"])
    return triplets_df

In [197]:
def get_prompt(graph, question, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01"):

    main_triplets_related_to_question = find_question_related_triplets(
        graph=graph,
        question=question,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_k=1000
    )

    additional_triplets_related_to_question = find_additional_related_triplets(
        graph=graph,
        question=question,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_entity_k=10,
        top_triplet_k=1000
    )

    final_triplets = pd.concat([main_triplets_related_to_question, additional_triplets_related_to_question], ignore_index=True)

    final_triplets = final_triplets.drop_duplicates(subset=["subject", "predicate", "object", "date", "start", "end", "speech_id"])
    final_triplets = final_triplets.sort_values(by=["date", "start"], ascending=[True, True]).reset_index(drop=False)

    final_triplets["index"] = final_triplets.index + 1

    main_triplets = final_triplets[final_triplets["information_type"] == "main"]
    additional_triplets = final_triplets[final_triplets["information_type"] == "additional"]

    prompt = QUESTION_ANSWER_PROMPT.format(
        question=question,
        main_facts=main_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records"),
        additional_facts=additional_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records")
    )

    return prompt, final_triplets

In [308]:
def get_fragment(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    start = int(row.start)
    end = int(row.end) + 1
    speech_id = str(int(row.speech_id) - 1)
    return ".".join(es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"].split(".")[start:end])

def get_speech(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    speech_id = str(int(row.speech_id) - 1)

    return es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"]

In [212]:
QUESTION_ANSWER_PROMPT = """
You are a political scientist model. You are given a research question and a set of facts from a knowledge graph.
Each of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.
Each of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.
The triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.
Each fact contains an index.
You can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".
Use just syntax with [index] to refer to the fact, do not use any additional words.

Facts are split into two categories: main and additional.
Main facts (most important) are directly related to the research question. 
Additional facts provide context or background information for entities present in the question.

Research question: {question}

Main Facts:
{main_facts}

Additional Facts:
{additional_facts}

Your task is to answer the research question based on the provided facts."
"""

In [213]:
QUESTION = """
What historical events does he use to legitimize actions toward Ukraine?
"""

START_DATE = "1999-01-01"
END_DATE = "2024-12-31"

PROMPT, final_triplets = get_prompt(
    graph=graph,
    question=QUESTION,
    score_threshold=0.5,
    start_date=START_DATE,
    end_date=END_DATE
)
print(f"Used main facts: {len(final_triplets[final_triplets['information_type'] == 'main'])}")
print(f"Used additional facts: {len(final_triplets[final_triplets['information_type'] == 'additional'])}")
PROMPT

Used main facts: 60
Used additional facts: 4


'\nYou are a political scientist model. You are given a research question and a set of facts from a knowledge graph.\nEach of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.\nEach of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.\nThe triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.\nEach fact contains an index.\nYou can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".\nUse just syntax with [index] to refer to the fact, do not use any additional words.\n\nFacts are split into two categories: main and additional.\nMain facts (most important) are directly related to the researc

In [ ]:
res = ollama_request(
    prompt=PROMPT, is_stream=False
)
 
display(Markdown(res))

In [322]:
display(Markdown(es_client.search(index=INDEX_NAME, query={"match": {"_id": 634}}, size=100)["hits"]["hits"][0]["_source"]["text"]))

Mister Federal Chancellor, Ladies and Gentlemen,  I cordially welcome the participants of the Third Petersburg Dialogue Public Forum. Our April meetings are becoming a good, decent tradition. This time they are being held in the year of the 300th anniversary of the foundation of St. Petersburg. We are sincerely happy that Germany will be one of the most active participants of the anniversary celebrations.  Such events as the forum in which we are today taking part are one more proof of the rapprochement of our countries. In the last few years it has been proceeding most intensively. And what is extremely important — relations between Russia and Germany are actively developing at the level of civilian and human contacts.  I am convinced: the development of European civilization largely depends on the degree of understanding between Russians and Germans. I think that the Petersburg Dialogue participants well feel this high bar which we call ”strategic partnership.“  In life, in everyday contacts between people such partnership is not only the similarity of interests and long-term aims. It presupposes a deep knowledge of each other, and requires mutual respect, equality and trust.  It is known that partner relations are properly built on three equivalent pillars. They are the political sphere, trade-and-economic cooperation and relations in the humanitarian and cultural fields.  The incompleteness or instability of any of these pillars can undermine the reliability of the whole construction. Therefore huge efforts have been exerted in the last few years to ensure that all the components of Russian-German relations develop consistently and dynamically. I see here also the weighty contribution of this forum.  Now I would like to briefly dwell on the main areas of the partnership of our countries.  The first is the interests of Russia and Germany in the field of politics.  One of the tasks here is the formation of an effective European security system. The stability of Europe in many respects is a synonym for the stability of our countries as well, and so to meet the certain new threats it is necessary, without doubt, to act together.  Now it is also important to emphasize our position on the most acute and crucial problem — the situation in Iraq. Military actions have now been continuing there for more than three weeks. Their results are known and they cause regret.  You know that both Moscow and Berlin came out for a political solution of the problem of Iraq. We now too are convinced of the futility of the military solution, and hold that the principal task is to return the settlement process to the UN framework as soon as possible.  Our countries can and should do everything possible to preserve a stable international legal system which is based on the supremacy of the UN. Mr. Federal Chancellor and I are unanimous in the understanding of the primacy of international law.  I must note that in the last few months Russia and Germany were very closely cooperating in the UN Security Council, were acting in one and the same system of foreign policy coordinates.  Dear Ladies and Gentlemen,  Not a single one of our meetings with the Federal Chancellor passes without discussing economic problems, which are today the pivot of bilateral relations.  To us, Germany is the chief foreign economic partner. This is confirmed by the steady tendency for the growth of exchanges of goods and the activity of German capital in Russia alike. In volume of investment the FRG continues to be the leader of this process.  There are other facts, however. Whereas Germany accounts for 15 percent of Russia's foreign trade turnover, our share of Germany's foreign trade is only two percent.  The reasons for this state of affairs are many. I think that here, at the forum, people are gathered who know them very well. In order to change the situation, officials and businesspeople all have to work together. Only a continual and mutually beneficial dialogue between the business circles of our countries will facilitate the search of new approaches and solutions.  I know that the establishment of uniform principles of accreditation for our countries will be discussed at the forum. In Russia such a system of confirmation of competence in the professional sphere is practically nonexistent. And the assistance of Germany, where it is already successfully operating, is undoubtedly useful. That approach will help solve economic tasks of a systemic character and ensure our country international recognition in this field.  Germany has also accumulated a considerable experience of the transfer of economic functions from the state to non-governmental organizations. For Russia this is also one of the most vital tasks. The very discussion of these questions is a big stride towards the reasonable organization of the economic life in our country.  It is obvious that in the sphere of economic cooperation there still exist many reserves and potentialities. That's why it is so important now to exchange views and experience, to determine the positions and to work out our common approaches.  Dear colleagues,  Briefly — on the cultural and humanitarian dimension of our relations. Russian and German cultures are absolutely self-sufficient. But at the same time they are mutually complementary. The mutual penetration of our two cultures has contributed to the enrichment of European civilization, and helped the overcoming of mutual distrust and historical reconciliation between our countries.  In February in Berlin a unique event was launched — Russian-German Cultural Meetings. Having begun as part of it, the Year of Russian Culture in Germany will give the citizens of the FRG an opportunity to discover for themselves their own Russia. We count on an equally large-scale German ”cultural presence“ in our country next year.  Here, within the native walls of St. Petersburg University for me, I consider it important to note the work of the youth section of the Petersburg Dialogue. It has become an important component of the operative mechanism of youth exchanges between our countries. As another element of this system we regard also the Baltic Star international festival, in which high school students will take part.  Taking this opportunity, I would like to invite high school students from Germany to take an active part in it. It is opening in May in St. Petersburg, will pass through Helsinki, Stockholm and Lubeck and will conclude in June in Kaliningrad.  Another effective instrument in the field of youth politics is the Russian-German Youth Exchange Council.  Thus we have established a quite good mechanism of youth interaction, which we intend to perfect in the future as well.  This will give young people an opportunity to know and understand each other and will create a reliable platform for our future cooperation.  Dear friends,  Within a fairly short time, the Petersburg Dialogue has found its audience and niche, has acquired authority and I think is completely ready for further independent development.  Such meetings provide a unique opportunity to gain insights into the planes of our relations which sometimes are outside the field of vision of the politicians and statesmen. Such human contacts strengthen and enrich relations between our countries, and I wholeheartedly would like to wish you further productive work.  Thanks very much.  I consider it my duty to explain my position on these problems. First — regarding the statements of the representatives of certain media that Russia, Germany and, perhaps, France will be the initiators of a ”new Yalta.“ I will not speak for Mr. Federal Chancellor, but this does not correspond to my perceptions of how the situation ought to be built in the formation of the security system in the world. There should be no separate actions here. We should discuss this problem together with all member countries of the United Nations. We should perfect the international security system that has evolved in recent years, adapt it to the new realities and in no way allow it to be destroyed. Simply we have nothing other in exchange.  We have to perfect it — yes! But not to destroy it under any circumstances. And to act together anyway. This is the first point.  Second — with regard to the thesis of what is good and what is bad in the Iraqi events. It's already good that the regime of Hussein has been removed. And we had always said this regime did not correspond to the present-day requirements and perceptions of human rights and democracy and that it was necessary to change and remove it.  After all, we did not say that we were sheltering it. We spoke of a different thing — that problems of this kind should not be solved by means of war, by military means. Countries which do not meet the Western standards of democracy form 80 percent in the world. So what, we shall war with all of them?  These standards have to come into being within the countries themselves. The peoples of these states alone have the right to decide their own destinies. That is why we are saying that the principle of sovereignty must be immutable.  And then, are most of these countries ready for adopting in their territories the Western standards of democracy? For in the Middle East, apart from Iraq, there are many other states. And what, is it proposed to war with them all? Certainly, this mad thought occurs to nobody.  We indicated by what means we should act to attain these noble objectives. As to what was there good and what was bad, the removal of the tyrannical regime is surely a plus. But, I repeat, the means and the human casualties, the humanitarian disaster, the destructions — these are the obvious negative consequences.  As are the problems which have been created in the sphere of the system of international law observance, its being shaken loose. This arouses concern. In our opinion, we should do everything possible to restore this system of values and together with our partners, with the United States and with Britain tackle the problems of strengthening the principles and foundations of international law.  Tomorrow at the law department of the University in the course of the international conference The International Security System: A Look at the Future we shall be able to once again have a discussion on this topic.

In [323]:
REFERENCE_NUM = 31

display(final_triplets[final_triplets["index"] == REFERENCE_NUM])#[["subject", "predicate", "object"]].reset_index(drop=True))

display(Markdown("### Reference Fragment"))
display(Markdown(get_fragment(final_triplets, REFERENCE_NUM)))

display(Markdown("### Reference Speech"))
display(Markdown(get_speech(final_triplets, REFERENCE_NUM)))


,index,subject,predicate,object,date,start,end,speech_id,information_type
30,31,vladimir_putin,supports,war_in_ukraine,2003-04-11,24,25,634,additional


### Reference Fragment

### Reference Speech

Today we had very positive, substantive and constructive talks with the President of Turkmenistan Saparmurat Niyazov. Our joint conclusion is that the Russian-Turkmen relations are steadily developing and have a good future. We have exchanged opinions on the whole range of issues pertaining to Russian-Turkmen ties and discussed in substance the implementation of the agreements reached during the visit of the Turkmen President to the Russian Federation last year. This meeting has a particular significance. The Treaty on Friendship and Cooperation between our countries has come into force. We firmly believe that it will provide a solid legal basis for Russian-Turkmen relations. We paid serious attention to trade and economic links. Some positive developments have already taken place in this sphere. We have signed an agreement on cooperation between Russia and Turkmenistan in the gas sector until 2028. This programme, coupled with the contract for the supply of Turkmen gas to Russia concluded between Gazprom and Turkmenneftegaz, will strengthen our interaction in other important sectors and areas.  I would like to note that under the agreement half of the price of gas will be paid for by supplies of Russian goods to Turkmenistan. In this way we will load our production capacity, create more jobs in the Russian Federation and supply equipment to Turkmenistan creating a good basis for the development of trade and economic links in many areas. We believe that the start of the work of the intergovernmental commission on economic cooperation will provide a good basis for the development of the great potential of Russian-Turkmen interaction. We are sure it will help us to solve the issues that have up until now hindered the development of our ties.  The security cooperation agreement signed today will make the efforts of our countries in counteracting the modern challenges and threats more systemic and effective. We are sure it will have a positive impact on the further development of the relations between Russia and Turkmenistan.  We have come to the conclusion that the agreement on dual citizenship has fulfilled its role and we have agreed to annul it. The majority of the people who wanted to move to the Russian Federation have done so, and a new law on migration and citizenship has come into force in Russia. We have agreed to build our relations with Turkmenistan on the basis of that law.  That is all I wanted to say at the beginning of our joint work. The President of Turkmenistan and I will have more opportunities to discuss the whole range of interaction in an informal atmosphere later today. 